In [2]:
!pip install catboost xgboost -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.7 MB/s eta 0:00:00


In [3]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor

In [4]:
df = pd.read_csv("EMS_Feature_Engineered.csv")

print(df.shape)

df.head()

(3629661, 29)


,initial_call_type,final_call_type,dispatch_response_seconds_qy,incident_response_seconds_qy,incident_travel_tm_seconds_qy,zipcode,borough,incident_dispatch_area,hour,month,...,closure_time,zipcode_frequency,borough_frequency,dispatch_area_frequency,initial_call_frequency,final_call_frequency,rush_hour,office_hours,call_borough,zip_hour
0,53.0,51.0,53,393.0,340.0,11365.0,3.0,26.0,0.0,1.0,...,-8276829.0,9311,743558,43405,437223,416373,0,0,337,4244
1,21.0,17.0,84,430.0,346.0,10457.0,0.0,1.0,0.0,1.0,...,3195.0,59102,854212,253459,188550,197297,0,0,180,2298
2,99.0,111.0,0,564.0,433.0,11224.0,1.0,6.0,0.0,1.0,...,6840.0,34013,1002290,150556,985,8795,0,0,536,3547
3,84.0,95.0,1485,1934.0,449.0,10033.0,2.0,21.0,0.0,1.0,...,2783.0,25033,873477,98891,33155,27167,0,0,470,744
4,20.0,16.0,39,1315.0,1276.0,11208.0,1.0,9.0,0.0,1.0,...,4144.0,51519,1002290,252294,3778,3513,0,0,176,3163


In [5]:
df = df.dropna(subset=["dispatch_response_seconds_qy"])

print(df.shape)

(3629661, 29)


In [6]:
duplicate_cols = [
    "Hour",
    "Month",
    "Weekday"
]

df.drop(columns=duplicate_cols, inplace=True, errors="ignore")

In [7]:
drop_cols = [

    "dispatch_response_seconds_qy",

    "incident_response_seconds_qy",

    "incident_travel_tm_seconds_qy",

    "travel_duration",

    "hospital_time",

    "closure_time"
]

X = df.drop(columns=drop_cols)

y = df["dispatch_response_seconds_qy"]

In [8]:
X_train, X_test, y_train, y_test = train_test_split(

    X,

    y,

    test_size=0.20,

    random_state=42

)

In [9]:
rf = RandomForestRegressor(

    n_estimators=50,

    max_depth=15,

    random_state=42,

    n_jobs=-1

)

rf.fit(X_train,y_train)

rf_pred = rf.predict(X_test)

rf_pred = np.maximum(rf_pred,0)

In [10]:
cat = CatBoostRegressor(

    iterations=500,

    learning_rate=0.05,

    depth=10,

    loss_function="RMSE",

    verbose=100,

    random_seed=42

)

cat.fit(X_train,y_train)

cat_pred = cat.predict(X_test)

cat_pred = np.maximum(cat_pred,0)

0:	learn: 1172.2674151	total: 1s	remaining: 8m 20s
100:	learn: 721.7292617	total: 1m 30s	remaining: 5m 56s
200:	learn: 703.8217730	total: 2m 57s	remaining: 4m 23s
300:	learn: 692.1342726	total: 4m 24s	remaining: 2m 54s
400:	learn: 683.5600398	total: 5m 52s	remaining: 1m 26s
499:	learn: 676.9449778	total: 7m 18s	remaining: 0us


In [11]:
xgb = XGBRegressor(

    n_estimators=500,

    learning_rate=0.05,

    max_depth=8,

    subsample=0.8,

    colsample_bytree=0.8,

    tree_method="hist",

    random_state=42,

    n_jobs=-1

)

xgb.fit(X_train,y_train)

xgb_pred = xgb.predict(X_test)

xgb_pred = np.maximum(xgb_pred,0)

In [12]:
def evaluate(name,y_true,pred):

    mae = mean_absolute_error(y_true,pred)

    rmse = np.sqrt(mean_squared_error(y_true,pred))

    r2 = r2_score(y_true,pred)

    print("="*50)

    print(name)

    print("MAE :",round(mae,2))

    print("RMSE :",round(rmse,2))

    print("R2 :",round(r2,4))

In [13]:
evaluate("Random Forest",y_test,rf_pred)

evaluate("CatBoost",y_test,cat_pred)

evaluate("XGBoost",y_test,xgb_pred)

Random Forest
MAE : 165.34
RMSE : 728.52
R2 : 0.6366
CatBoost
MAE : 165.92
RMSE : 713.92
R2 : 0.651
XGBoost
MAE : 165.24
RMSE : 709.64
R2 : 0.6552


In [14]:
importance = pd.DataFrame({

    "Feature":X.columns,

    "Importance":xgb.feature_importances_

})

importance = importance.sort_values(

    by="Importance",

    ascending=False

)

importance.head(20)

,Feature,Importance
11,dispatch_delay,0.296566
10,incident_duration,0.271813
5,hour,0.034438
17,initial_call_frequency,0.033316
21,call_borough,0.028935
13,night_shift,0.028579
9,Year,0.027506
15,borough_frequency,0.024185
1,final_call_type,0.022556
22,zip_hour,0.022547


In [15]:
joblib.dump(

    xgb,

    "dispatch_time_model.pkl"

)

print("Dispatch Time Model Saved Successfully!")

Dispatch Time Model Saved Successfully!


In [18]:
from google.colab import files

files.download("dispatch_time_model.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
print(type(xgb))

<class 'xgboost.sklearn.XGBRegressor'>


In [17]:
import joblib

model = joblib.load("dispatch_time_model.pkl")
print("Loaded Successfully!")

Loaded Successfully!


In [2]:
import xgboost
import sklearn
import joblib

print(xgboost.__version__)
print(sklearn.__version__)

model = joblib.load("../models/dispatch_time_model (1).pkl")
print("✅ Loaded Successfully")

3.3.0
1.6.1


XGBoostError: input stream corrupted